In [1]:
import numpy as np
import pandas as pd

from exerpy import ExergyAnalysis
from exerpy.analyses import _load_json

In [2]:
def diff_between_simulators(testcase):
    simulator_results = []
    for path in testcase.values():
        contents = _load_json(path)
        if "settings" not in contents:
            contents["settings"] = {}

        simulator_results += [
            ExergyAnalysis.from_json(path, **contents["settings"])
        ]

    sim1 = simulator_results[0]
    sim2 = simulator_results[1]

    columns = ["m", "p", "T"]
    if sim1.chemExLib is not None:
        columns.append("e_CH")
    if sim1.split_physical_exergy:
        columns.append("e_M")
        columns.append("e_T")
    else:
        columns.append("e_PH")

    df_sim1 = pd.DataFrame.from_dict(
        sim1._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")
    df_sim2 = pd.DataFrame.from_dict(
        sim2._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")

    overlapping_index = list(
        set(df_sim1.index.tolist()) & set(df_sim2.index.tolist())
    )
    df_sim1 = df_sim1.loc[overlapping_index].round(6)
    df_sim2 = df_sim2.loc[overlapping_index].round(6)

    # inf means that sim2 has 0 value, comparison does not make sense there
    # and sometimes there seem to be NaN values in the dataframes, those are
    # removed as well
    diff = ((df_sim1 - df_sim2) / df_sim2).abs().replace(np.inf, 0).fillna(0)
    # exergies near the ambient state are close to zero, so differences below 1 J/kg are round-off
    exergies = [column for column in columns if column.startswith("e_")]
    diff[exergies] = diff[exergies].mask((df_sim1 - df_sim2)[exergies].abs() < 1.0, 0)
    return diff

In [3]:
testcase = {
    "ebsilon": "ccpp_ebs.json",
    "tespy": "ccpp_tespy.json"
}
diff_between_simulators(testcase)


,m,p,T,e_CH,e_M,e_T
4,0.000204,0.000000e+00,1.124267e-08,0.001249,0.000542,0.000217
1,0.000220,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
9,0.000055,0.000000e+00,1.241861e-04,0.000000,0.000000,0.000099
5,0.000204,5.222437e-10,1.203114e-04,0.001249,0.000000,0.000394
16,0.000016,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
18,0.000073,0.000000e+00,2.257194e-09,0.000000,0.000000,0.000000
22,0.000055,1.072926e-10,1.882696e-09,0.000000,0.000000,0.000000
10,0.000096,0.000000e+00,1.324507e-04,0.000000,0.000000,0.000085
20,0.000055,0.000000e+00,2.207369e-09,0.000000,0.000000,0.000000
12,0.000016,0.000000e+00,1.324507e-04,0.000000,0.000000,0.000085


In [4]:
testcase = {
    "tespy": "ccpp_tespy.json",
    "aspen": "ccpp_aspen.json"
}
diff_between_simulators(testcase)

,m,p,T,e_CH,e_M,e_T
4,0.000415,0.0,7.026666e-09,0.000222,0.001090,0.000880
1,0.000418,0.0,0.000000e+00,0.000000,0.000000,0.000000
9,0.000099,0.0,3.252940e-05,0.000000,0.000000,0.000025
5,0.000415,0.0,3.151899e-05,0.000222,0.000000,0.000234
16,0.000036,0.0,3.267715e-09,0.000000,0.000000,0.000000
18,0.000019,0.0,0.000000e+00,0.000000,0.000000,0.000000
22,0.000099,0.0,6.024663e-06,0.000000,0.000000,0.000025
10,0.000704,0.0,3.467468e-05,0.000000,0.000000,0.000022
20,0.000099,0.0,8.299293e-05,0.000000,0.000000,0.000410
12,0.000036,0.0,3.467468e-05,0.000000,0.000000,0.000022


In [5]:
testcase = {
    "ebsilon": "ccpp_ebs.json",
    "aspen": "ccpp_aspen.json"
}
diff_between_simulators(testcase)

,m,p,T,e_CH,e_M,e_T
4,0.000211,0.000000e+00,4.216000e-09,0.001026,0.001633,0.000663
1,0.000199,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
19,0.000075,0.000000e+00,3.355738e-06,0.000000,0.000000,0.000025
9,0.000154,0.000000e+00,1.567114e-04,0.000000,0.000000,0.000125
16,0.000052,0.000000e+00,3.267715e-09,0.000000,0.000000,0.000000
5,0.000211,5.222437e-10,1.518266e-04,0.001026,0.000000,0.000160
18,0.000092,0.000000e+00,2.257194e-09,0.000000,0.000000,0.000000
22,0.000154,1.072926e-10,6.022781e-06,0.000000,0.000000,0.000025
10,0.000800,0.000000e+00,1.671208e-04,0.000000,0.000000,0.000107
20,0.000154,0.000000e+00,8.299072e-05,0.000000,0.000000,0.000410


In [6]:
# Component results stored by the example scripts ccpp_ebs.py, ccpp_tespy.py and ccpp_aspen.py
results = {
    simulator: pd.read_csv(f"ccpp_components_{simulator.lower()}.csv", index_col=0).set_index("Component")
    for simulator in ("Ebsilon", "TESPy", "Aspen")
}

# Components the simulators model differently are compared as one group
groups = {
    "MIX": "DEA",  # Ebsilon and Aspen mix the feedwater before the deaerator
    "dea steam inlet valve": "DEA",  # TESPy throttles the deaerator steam
    "drum": "EVA",  # TESPy models the drum and its pump separately
    "drum pump": "EVA",
    "drum pump motor": "EVA",
    "PUMP1-MOTOR": "MOT1",  # Aspen names of the motors
    "PUMP2-MOTOR": "MOT2",
}

# Exergy destruction in kW of the components found in at least two models, and the deviations from Ebsilon
E_D = (
    pd.DataFrame({simulator: df["E_D [kW]"] for simulator, df in results.items()})
    .drop("TOT")
    .rename(index=groups)
    .groupby(level=0)
    .sum(min_count=1)
    .dropna(thresh=2)
)
reference = E_D["Ebsilon"].where(E_D["Ebsilon"].abs() > 1e-3)  # no relative deviation below 1 W
for simulator in ("TESPy", "Aspen"):
    E_D[f"{simulator} vs Ebsilon [%]"] = (E_D[simulator] / reference - 1) * 100
E_D.sort_values("Ebsilon", ascending=False).round(2)

,Ebsilon,TESPy,Aspen,TESPy vs Ebsilon [%],Aspen vs Ebsilon [%]
Component,,,,,
CC,195881.72,195534.64,195419.14,-0.18,-0.24
GT,15986.83,15984.36,15986.42,-0.02,-0.00
EVA,11644.30,11650.64,11645.90,0.05,0.01
COMP,10958.82,10953.65,10953.33,-0.05,-0.05
SH,4343.62,4345.27,4345.13,0.04,0.03
GEN1,3777.78,3777.77,3777.75,-0.00,-0.00
DEA,3182.87,3183.01,3182.49,0.00,-0.01
ST2,2998.26,2998.48,2998.44,0.01,0.01
COND,2335.88,2335.82,2335.89,-0.00,0.00


In [7]:
# Results of the whole plant
columns = ["E_F [kW]", "E_P [kW]", "E_D [kW]", "E_L [kW]", "epsilon [%]"]
pd.DataFrame({simulator: df.loc["TOT", columns] for simulator, df in results.items()}).round(3)

,Ebsilon,TESPy,Aspen
E_F [kW],641594.990,641204.900,641058.919
E_P [kW],339699.736,339700.685,339700.930
E_D [kW],254471.752,254124.950,254005.617
E_L [kW],47423.503,47379.265,47352.373
epsilon [%],52.946,52.978,52.991
